In [1]:
!nvidia-smi

Fri May  8 19:09:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!/usr/local/cuda/bin/nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [3]:
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmp6tn6wmxk".


In [4]:
%%writefile vector_multiplication.cu

#include <iostream>
#include <cuda.h>
using namespace std;

__global__ void multiply(int *a, int *b, int *c, int n) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < n && col < n) {
        int sum = 0;
        for (int k = 0; k < n; k++)
            sum += a[row*n + k] * b[k*n + col];

        c[row*n + col] = sum;
    }
}

int main() {
    int n;
    cin >> n;

    int size = n * n * sizeof(int);

    // Host memory (C++ style)
    int *a = new int[n*n];
    int *b = new int[n*n];
    int *c = new int[n*n];

    int *d_a, *d_b, *d_c;

    // Input
    for (int i = 0; i < n*n; i++) cin >> a[i];
    for (int i = 0; i < n*n; i++) cin >> b[i];

    // Allocate GPU memory
    cudaMalloc((void**)&d_a, size);
    cudaMalloc((void**)&d_b, size);
    cudaMalloc((void**)&d_c, size);

    // Copy data to GPU
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // Define block and grid
    dim3 block(16, 16);
    dim3 grid((n + 15) / 16, (n + 15) / 16);

    // Launch kernel
    multiply<<<grid, block>>>(d_a, d_b, d_c, n);
    cudaDeviceSynchronize();

    // Copy result back
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    // Output matrix
    for (int i = 0; i < n; i++) {
        for (int j = 0; j < n; j++)
            cout << c[i*n + j] << " ";
        cout << endl;
    }

    // Free memory
    delete[] a;
    delete[] b;
    delete[] c;

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}


Writing vector_multiplication.cu


In [5]:
!nvcc -arch=sm_75 vector_multiplication.cu -o vector_multiplication

In [6]:
!echo "2 1 2 3 4 5 6 7 8" | ./vector_multiplication


19 22 
43 50 
